<a href="https://colab.research.google.com/github/DevisriprasadSoma/MLflyrank/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DevisriprasadSoma/MLflyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — Content Lifecycle

The paper reports that growing pages are younger on average than declining pages: about 185 days versus 228 days. It also reports that average word count is almost the same between the two groups, so age appears more strongly associated with the observed difference than length.

**Methodology question:** How exactly is the growing-versus-declining label constructed, and does the validation design ensure that the age comparison is not partly reflecting differences in the observation window or other page-level factors? A useful check would be to confirm that the label is defined before the comparison and that the validation uses held-out observations appropriate to the claim.

### Finding 2 — Freshness Multiplier

The paper reports a 5.4:1 growth-to-decline ratio for pages updated 31–90 days ago and a separate 365+ page cohort where recently refreshed pages showed higher health and impressions than older stale pages. The paper also notes that some very old buckets are small, which makes those estimates harder to interpret.

**Methodology question:** For the reported refresh lift, is the comparison based on a clearly defined pre-refresh label and an outcome window after refresh, or is it a cross-sectional comparison of already-refreshed versus stale pages? A stronger causal interpretation would require careful separation of the pre-period and outcome period and controls for differences between pages that were refreshed and pages that were not.

### Overall reflection

These questions are intended as constructive methodology checks rather than challenges to the paper's findings. The paper itself states that its main results are observational patterns and should not be treated as proof of cause and effect.


In [7]:
print("Section 1 completed.")
print("Two paper findings reviewed: Content Lifecycle and Freshness Multiplier.")
print("Methodology questions focus on label construction and validation design.")

Section 1 completed.
Two paper findings reviewed: Content Lifecycle and Freshness Multiplier.
Methodology questions focus on label construction and validation design.


## 2. My model under an honest split (before/after)

In ML-08, the Logistic Regression model was evaluated with a stratified 80/20 holdout and achieved a ROC-AUC of 0.6777 and Average Precision of 0.6945.

For this audit, I repeat the experiment using a client-grouped split. Pages belonging to the same `client_id` are kept within either the training or test set, reducing the possibility that client-specific patterns are shared across both sets.

I use the same target definition and the same 19 model features as ML-08. The purpose of the comparison is to measure how the model's ranking performance changes under a stricter validation design.

The grouped result is treated as a more conservative development estimate. Any difference between the random holdout and grouped result is interpreted as validation sensitivity, not as proof of model failure or success in production.


In [8]:
# ============================================
# SECTION 2 — BEFORE / AFTER VALIDATION
# ============================================

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# --------------------------------------------
# Load the same dataset used in ML-08
# --------------------------------------------

repo = "/content/flyrank-ml-internship-starter"
raw_path = os.path.join(
    repo,
    "data/raw/content_refresh_anonymized.csv"
)

if not os.path.exists(raw_path):
    !git clone -q https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

df = pd.read_csv(raw_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))

# --------------------------------------------
# Create the same target as ML-08
# --------------------------------------------

df["target"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# --------------------------------------------
# Same 19 features as ML-08
# --------------------------------------------

feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "char_count",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

X = df[feature_cols].copy()
y = df["target"].copy()
groups = df["client_id"].copy()

for col in feature_cols:
    X[col] = pd.to_numeric(
        X[col],
        errors="coerce"
    ).fillna(0)

# --------------------------------------------
# BEFORE — ML-08 stratified holdout
# --------------------------------------------

X_train_before, X_test_before, y_train_before, y_test_before = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model_before = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model_before.fit(
    X_train_before,
    y_train_before
)

scores_before = model_before.predict_proba(
    X_test_before
)[:, 1]

auc_before = roc_auc_score(
    y_test_before,
    scores_before
)

ap_before = average_precision_score(
    y_test_before,
    scores_before
)

# --------------------------------------------
# AFTER — client-grouped holdout
# --------------------------------------------

group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    group_split.split(
        X,
        y,
        groups=groups
    )
)

X_train_after = X.iloc[train_idx]
X_test_after = X.iloc[test_idx]

y_train_after = y.iloc[train_idx]
y_test_after = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

model_after = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model_after.fit(
    X_train_after,
    y_train_after
)

scores_after = model_after.predict_proba(
    X_test_after
)[:, 1]

auc_after = roc_auc_score(
    y_test_after,
    scores_after
)

ap_after = average_precision_score(
    y_test_after,
    scores_after
)

# --------------------------------------------
# Check client separation
# --------------------------------------------

overlap = len(
    set(groups_train) & set(groups_test)
)

# --------------------------------------------
# Before / After table
# --------------------------------------------

results = pd.DataFrame({
    "Validation": [
        "ML-08 Stratified Holdout",
        "ML-09 Client-Grouped Holdout"
    ],
    "Train Rows": [
        len(X_train_before),
        len(X_train_after)
    ],
    "Test Rows": [
        len(X_test_before),
        len(X_test_after)
    ],
    "ROC-AUC": [
        round(auc_before, 4),
        round(auc_after, 4)
    ],
    "Average Precision": [
        round(ap_before, 4),
        round(ap_after, 4)
    ]
})

print("\nBEFORE / AFTER VALIDATION")
print("-------------------------")
display(results)

print("\nClient overlap between train/test:", overlap)

print(
    "\nROC-AUC change:",
    round(auc_after - auc_before, 4)
)

print(
    "Average Precision change:",
    round(ap_after - ap_before, 4)
)

Rows: 30000
Columns: 44

BEFORE / AFTER VALIDATION
-------------------------


,Validation,Train Rows,Test Rows,ROC-AUC,Average Precision
0,ML-08 Stratified Holdout,24000,6000,0.6777,0.6945
1,ML-09 Client-Grouped Holdout,23837,6163,0.5950,0.5939



Client overlap between train/test: 0

ROC-AUC change: -0.0826
Average Precision change: -0.1007


### Before/after interpretation

The original ML-08 stratified holdout produced a ROC-AUC of 0.6777 and Average Precision of 0.6945. Under the stricter client-grouped validation, ROC-AUC decreased to 0.5950 and Average Precision decreased to 0.5939.

The ROC-AUC change was -0.0826 and the Average Precision change was -0.1007. The train/test client overlap was zero, so the grouped split successfully tested the model on clients that were not present during training.

This measured drop suggests that part of the original model performance may be related to client-specific patterns. The grouped result is therefore a more conservative estimate of how well the model may generalize to unseen clients. These results are development evidence and should not be interpreted as production performance.


## 3. Leakage audit

I audited the final 19 model features against the known label-derived fields.

The model features do not include `trend_direction`, `trend_pct`, or `target` as inputs. These fields are used to define or describe the outcome and would create leakage if supplied to the model.

I also excluded the last-30-day versus previous-30-day trend fields used to derive the decline outcome from the feature set. The remaining features describe the content's observed visibility, traffic, engagement, freshness, and content characteristics.

The audit therefore finds no known label-derived fields in the final model feature set. This is a feature-level leakage check; it does not prove that every possible source of bias or leakage is absent.


In [9]:
# ============================================
# SECTION 3 — LEAKAGE AUDIT
# ============================================

label_derived = [
    "trend_direction",
    "trend_pct",
    "target"
]

feature_set = set(feature_cols)

print("LEAKAGE AUDIT")
print("-------------")

print("\nModel features:")
for feature in feature_cols:
    print("-", feature)

print("\nKnown label-derived fields:")
for field in label_derived:
    print("-", field)

print("\nLabel-derived fields used as model features:")

leakage_found = feature_set.intersection(
    set(label_derived)
)

print(leakage_found)

print("\nFuture/outcome fields excluded:")
excluded = [
    "trend_direction",
    "trend_pct"
]

for field in excluded:
    print("-", field)

print("\nLeakage audit result:")

if len(leakage_found) == 0:
    print("PASS — no known label-derived fields are present in the 19 model features.")
else:
    print("REVIEW — possible leakage fields detected.")

LEAKAGE AUDIT
-------------

Model features:
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- days_with_impressions
- days_with_sessions
- content_age_days
- days_since_last_update
- word_count
- char_count
- ctr
- avg_position
- engagement_rate
- scroll_rate
- ai_traffic_pct

Known label-derived fields:
- trend_direction
- trend_pct
- target

Label-derived fields used as model features:
set()

Future/outcome fields excluded:
- trend_direction
- trend_pct

Leakage audit result:
PASS — no known label-derived fields are present in the 19 model features.


## 4. Claim rewrite

### Original claim

The Logistic Regression model provides additional ranking signal over the Week-4 rule-based baseline and can help identify content likely to decline.

### Revised claim

On the development holdout, Logistic Regression showed higher ranking performance than the Week-4 baseline, with ROC-AUC of 0.6777 versus 0.5787 and Average Precision of 0.6945 versus 0.5699.

However, performance decreased under client-grouped validation to a ROC-AUC of 0.5950 and Average Precision of 0.5939. This suggests that some of the original performance may depend on client-specific patterns.

Therefore, the measured results provide directional evidence that Logistic Regression can add decision-support signal on this development dataset, but they do not establish reliable future or production performance.


In [10]:
print("Claim rewrite completed.")
print("Original validation: ROC-AUC 0.6777, AP 0.6945")
print("Client-grouped validation: ROC-AUC 0.5950, AP 0.5939")
print("Final claim uses measured, directional, and decision-support language.")

Claim rewrite completed.
Original validation: ROC-AUC 0.6777, AP 0.6945
Client-grouped validation: ROC-AUC 0.5950, AP 0.5939
Final claim uses measured, directional, and decision-support language.


## Self-check

Before you submit, confirm each line honestly:

* ☑ Every section above is filled — markdown thinking AND the code that backs it
* ☑ The notebook runs top to bottom with no errors (Runtime → Run all)
* ☑ No client names, URLs, or private queries anywhere
* ☑ My claims use careful words: observed, measured, directional, decision-support
* ☑ Committed to my repo under `work/notebooks/` — then submit my repo URL on the card. Done.
